In [0]:
#Broadcast variable = read‑only shared variable sent to all executors only once.
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("BroadcastExample").getOrCreate()

# Small lookup table
lookup = {
    "A": "Apple",
    "B": "Banana",
    "C": "Cherry"
}

broadcast_lookup = spark.sparkContext.broadcast(lookup)

# Big dataset
data = ["A", "B", "C", "A", "B", "C"]
rdd = spark.sparkContext.parallelize(data)

def map_values(x):
    fruit = broadcast_lookup.value.get(x)
    return (x, fruit)

result = rdd.map(map_values).collect()
print(result)


#Expected Output Need to map the value for ["A", "B", "C", "A", "B", "C"] 
#(["A", "Apple"], ["B", "Banana"], ["C", "Cherry"], ["A", "Apple"], ["B", "Banana"], ["C", "Cherry"])

In [0]:
# Accumulator = a write‑only shared variable used for counters or sums across executors.  
# Workers can add to it, but cannot read it.
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("AccumulatorExample").getOrCreate()
acc = spark.sparkContext.accumulator(0)
data = [1, 2, 3, 4, 5]
rdd = spark.sparkContext.parallelize(data)

def add_if_even(x):
    if x % 2 == 0:
        acc.add(x)
    return x

rdd.map(add_if_even).collect()
print("Total even numbers:", acc.value)

In [0]:
# UDF = Your own Python function used inside Spark SQL/DataFrame operations.  
# Used when Spark built‑in functions are not enough.

# ⚠️ UDFs are slower because they break Spark’s optimization.



from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, lit,current_date
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.appName("UDFExample").getOrCreate()
df = spark.createDataFrame([(1,), (2,), (3,)], ["num"])

def square(x):
    return x * 2

square_udf = udf(square)
df2 = df.withColumn("double_num", square_udf(df.num))
df2 = df2.withColumn("Dept", lit("Sales"))
df2=df2.withColumn("Currentdate",current_date())
display(df2)

num,double_num,Dept,Currentdate
1,2,Sales,2026-05-11
2,4,Sales,2026-05-11
3,6,Sales,2026-05-11


In [0]:
from pyspark.sql import Row

# Sample DataFrames with different columns
df1 = spark.createDataFrame([Row(id=1, name="Alice"), Row(id=2, name="Bob")])
df2 = spark.createDataFrame([Row(id=3, age=30), Row(id=4, age=25)])


display(df1)
display(df2)
# Union by name, allowing missing columns
df_union = df1.unionByName(df2, allowMissingColumns=True)
display(df_union)

id,name
1,Alice
2,Bob


id,age
3,30
4,25


id,name,age
1,Alice,null
2,Bob,null
3,null,30
4,null,25
